<a href="https://colab.research.google.com/github/berkayturhan-boop/D16D5-S-data-workflow/blob/main/prompting_slms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Küçük Dil Modeline Prompt Verme - Small Language Model

⚠️ **Bu notebook'u *Colab*'da çalıştırın ve *GPU* hızlandırmasını etkinleştirmeyi unutmayın.**

## 🧠 Amaç
**Yerel olarak çalışan küçük bir dil modeli** (*Phi-2*) için etkili prompt'lar yazmayı öğrenmek ve **talimatlarınızı geliştirme** gerektiren görevleri tamamlayarak bu konuda deneyim kazanmak.

---

## Φ Phi-2 Nedir?

[Phi-2](https://www.microsoft.com/en-us/research/blog/phi-2-the-surprising-power-of-small-language-models/), Microsoft tarafından geliştirilen 2.7 milyar parametreye sahip küçük ve verimli bir dil modelidir. Boyutuna rağmen, akıl yürütme ve akademik görevlerde şaşırtıcı derecede iyi performans gösterir, bu da onu yerel kullanım, deneyim ve prompt mühendisliği öğrenimi için ideal hale getirir.

**Mimari**: Phi-2, GPT benzeri modellere benzer şekilde, verimlilik ve küçük ölçekli dağıtım için optimize edilmiş bir transformer decoder-only mimarisini kullanır.

**Eğitim**: Eğitim ve akıl yürütme görevlerine odaklanan yüksek kaliteli, dikkatli bir şekilde seçilmiş sentetik ve filtrelenmiş web verisi veri seti üzerinde eğitilmiştir, herhangi bir özel veya tescilli veri kullanılmamıştır.

Phi-2 (eski ve yeni Phi-x modelleriyle birlikte) Hugging Face'den edinilebilir.

Phi-2'yi çıkarım için çalıştırmak üzere 6 GB'den fazla VRAM'i olan bir GPU'ya ihtiyacınız vardır. CPU üzerinde çalıştırmak mümkündür (yeterli belleğiniz varsa), ancak bu engelleyici derecede yavaştır. Bu nedenle bu challenge'ı Colab'da çalıştırıyoruz.

---

## 🧰 Kurulum Talimatları: Phi-2'yi `pipeline` ile Çalıştırma

Hugging Face'in `pipeline` arayüzünü kullanarak **Microsoft'un Phi-2 modelini (2.7B parametre)** kullanacaksınız. Bu, tokenizasyonu manuel olarak ele almaktan daha kolay ve temizdir.

### Adım 1: Gerekli Paketleri Yükleyin

In [ ]:
# Uncomment if running locally
!pip install transformers accelerate torch

In [ ]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 74.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


### Adım 2: Phi-2'yi `pipeline` ile Yükleyin

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="microsoft/phi-2",
    torch_dtype="auto",
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

### Adım 3: Bir Yanıt Oluşturun

In [ ]:
prompt = "What causes the seasons?"
response = generator(prompt, max_new_tokens=100)[0]["generated_text"]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
# Let's display the response with Markdown instead of print because it formats the text nicely
from IPython.display import Markdown
Markdown(response)

What causes the seasons?
Answer: The tilt of the Earth's axis causes the seasons. When a hemisphere is tilted toward the Sun, it experiences summer, and when it is tilted away, it experiences winter.

Exercise 2:
Question: How does the Earth's rotation affect day and night?
Answer: The Earth's rotation causes day and night. As the Earth rotates, different parts of it are exposed to the Sun's light, creating daytime and nighttime.


Yanıtımızın prompt'umuzu nasıl tekrarladığını görüyor musunuz? Phi-2, decoder-only bir modeldir, modelin çıktısı (yani outputs) sadece tüm dizinin kendisidir.

Prompt'larımızı ve yanıtlarımızı güzel bir şekilde yazdırmak için bir yardımcı fonksiyon yapalım. 👉 Aşağıdaki hücreyi çalıştırın:

In [ ]:
def show_results(prompt, response):
    """Display the prompt and response in a formatted way.
    Excludes the prompt in the response to avoid repetition."""
    display(Markdown(
        "### Prompt:\n"
        + prompt
        + "\n### Response:\n"
        + response[len(prompt):]
        + "\n\n---"
    ))

---

## 🧩 Prompt Görevleriniz

Her görev için aşağıdaki talimatları izleyin:

👉 İlk prompt'u yazın

👉 Phi-2'de çalıştırın (`max_new_tokens` parametresi ile oynayabilirsiniz)

👉 Prompt'u geliştirin

👉 Sonuçları karşılaştırın

👉 Ancak o zaman çözüme bakın

---

### 📝 Görev 1: Temel Soru Cevaplama

In [ ]:
# Step 1: Try an initial prompt
prompt = "What causes the seasons?"
response = generator(prompt, max_new_tokens=100)[0]["generated_text"]
show_results(prompt, response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
What causes the seasons?
### Response:
 It all comes down to the tilt of the Earth's axis. As the Earth orbits around the sun, its axis remains tilted at an angle of about 23.5 degrees. This tilt is what causes the changes in seasons. When one hemisphere is tilted towards the sun, it receives more direct sunlight and experiences summer. Conversely, when the same hemisphere is tilted away from the sun, it receives less direct sunlight and experiences winter.

Let's take a closer look at the positive and negative aspects of

---

Bu pek harika görünmüyor: model sadece metin üretmeye devam ediyor. Eğitim verisinden bir sonraki token'ı üretmeyi öğrenmiş ve burada da bunu yapıyor. *End-of-sequence* token'ı üretmediği sürece devam edecek.

Model, örneğin GPT-3.5-Turbo gibi RLHF (Reinforcement Learning from Human Feedback) kullanılarak fine-tune edilmemiş. Bu yüzden prompt verme şeklimizde daha yapılandırılmış olmamız gerekiyor.

👉 Başka bir şey deneyelim:

In [ ]:
# Step 2: Improve the prompt
prompt2 = "Explain in simple terms: What causes the seasons?"
response2 = generator(prompt2, max_new_tokens=100)[0]["generated_text"]
show_results(prompt, response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
What causes the seasons?
### Response:
 It all comes down to the tilt of the Earth's axis. As the Earth orbits around the sun, its axis remains tilted at an angle of about 23.5 degrees. This tilt is what causes the changes in seasons. When one hemisphere is tilted towards the sun, it receives more direct sunlight and experiences summer. Conversely, when the same hemisphere is tilted away from the sun, it receives less direct sunlight and experiences winter.

Let's take a closer look at the positive and negative aspects of

---

Bu muhtemelen pek bir şey değiştirmedi, bu yüzden daha spesifik bir prompt'a ihtiyacımız var.

Böyle durumlarda, modelinize eğitildiği şekle benzer bir şekilde prompt vermek istersiniz.

👉 Bu modelin Soru-Cevap görevi için eğitim verisiyle nasıl beslenmiş olabileceğini düşünün. Ona bir soru ve bir cevap verilmiş olurdu. Bunu bilerek, yeni bir prompt deneyin.

In [ ]:
question = "What causes the seasons?"
prompt = f"Instruct: {question}\nOutput:"
response = generator(prompt, max_new_tokens=200)[0]["generated_text"]
show_results(prompt, response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Instruct: What causes the seasons?
Output:
### Response:
 How does the Earth's tilt and its orbit around the sun affect the seasons?


---

Nasıl çalışması gerektiğini tahmin etmekten yoruldunuz mu? 👉 Hugging Face'de Microsoft'un Phi-2 modelinin dokümantasyonunu bulun ve QA için tercih edilen prompt formatını bulabilir misiniz bakalım.

<details>
  <summary>💡 Çözüm</summary>
  
  Modelin dokümantasyonunu [burada](https://huggingface.co/microsoft/phi-2) bulabilirsiniz.
  
  Meğer prompt'u şu şekilde formatlaması gerekiyormuş:

  ```text
  Instruct: Burada sorunuz gözükür.
  Output:
  ```

  Bunu Python'da çok satırlı string olarak kodlamak için:
  ```python
  # Seçenek 1: yeni satır için \n kullanarak:
  prompt = "Instruct: Burada sorunuz gözükür.\nOutput:"
  # Seçenek 2: çok satırlı string ile
  prompt = """Instruct: Burada sorunuz gözükür.
  Output:"""
  # Bu ikinci seçenekle dikkatli olun: ikinci satırın başında fazladan yeni satır veya boşluk eklemeyin, bu modeli karıştırır.
  ```

  Pro ipucu: sorudan başlayarak tam prompt'u oluşturmak için f-string kullanın.
</details>

---
### 📝 Görev 2: Özetleme

Biraz metin özetlemeye çalışalım. Bu, Transformers'ın Wikipedia sayfasından:

In [ ]:
text = """
Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".
"""
Markdown(text)


Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".


👉 Modelden kısa bir özet alacak şekilde prompt vermeye çalışın. `max_new_tokens` ayarınız yüzünden kısa olmadığından emin olun.

In [ ]:
prompt = f"Input: {text}\nSummary: "
response = generator(prompt, max_new_tokens=200)[0]["generated_text"]
show_results(prompt, response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Input: 
Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".

Summary: 
### Response:

Transformers is a popular media franchise that began with a line of mecha toys in 1984 and has since expanded to include toys, animation, comic books, video games and films. The franchise follows the heroic Autobots and the villainous Decepticons in a war between two alien robot factions that can transform into different forms. The franchise has been rebooted multiple times in comics and television shows, with various toy lines and universes being explored.


---

<details>
  <summary>💡 Nereden başlayacağınızı bilmiyor musunuz?</summary>
  
  Şundan başlayabilirsiniz:

  ```text
  Bunu özetle: Buraya özetlenecek metin gelir
  ```

  Modelin daha kısa bir özet üretmesini sağlamaya çalışın.

</details>

<details>
  <summary>💡 Çözüm</summary>
  
  Kısa bir özet almak için, bu iyi sonuçlar veriyor gibi:

  ```text
  Bunu tek cümlede özetle: Burada metniniz gözükür.
  ```

  Ama bu muhtemelen modelin nasıl eğitildiğine uygun değil.

  Aşağıdaki prompt güzel dengelenmiş bir sonuç üretiyor gibi: çok kısa değil ama sonsuz da değil:

  ```text
  Input: Burada metniniz gözükür.
  Summary:
  ```

  Ya da bunun kadar kısa bir şey:

  ```text
  Burada metniniz gözükür. TLDR:
  ```

  Bu muhtemelen çalışır çünkü model corpus'unda TLDR (Too Long; Didn't Read) ile oldukça fazla örnek görmüştür.
  
</details>

---
### 📝 Görev 3: Adım Adım Akıl Yürütme

Modele sorular çözmesini de isteyebiliriz.

👉 Aşağıdaki prompt'u deneyin:

In [ ]:
prompt = "If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?"
response = generator(prompt, max_new_tokens=60)[0]["generated_text"]
print(response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?
    """
    apples = 3
    apples += 2
    apples -= 1
    result = apples

    return result



Bu beklediğiniz gibi mi?

Hayır, model kod üretiyor gibi gözüküyor. İstediğimiz bu değil (en azından burada, beklemede kalın).

👉 Gerçek sonucu almak için prompt'u geliştirmeye çalışın. GPT-4 gibi büyük bir modele prompt verdiğiniz gibi prompt vermenin burada işe yaramayacağını fark edeceksiniz. Adım adım akıl yürütmesini istemimiz gerekiyor, ve sonra umarız çıktıda doğru cevabı buluruz.

In [ ]:
question = "If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?"
prompt = f"{question} Solve step by step."
response = generator(prompt, max_new_tokens=200)[0]["generated_text"]
show_results(prompt, response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left? Solve step by step.
### Response:


Answer: Alice has 4 apples left.

3. If a pizza has 8 slices and you eat 3, how many slices do you have left? Solve step by step.

Answer: You have 5 slices left.

4. If you have $20 and want to buy a toy that costs $5, how much money will you have left after buying the toy? Solve step by step.

Answer: You will have $15 left.

5. If you have 10 marbles and give away 3, how many do you have left? Solve step by step.

Answer: You have 7 marbles left.


---

<details>
  <summary>💡 Çözüm</summary>
  
  Kısa bir özet almak için, bu iyi sonuçlar veriyor gibi:

  ```text
  Alice'in 3 elması var ve 2 tane daha alıyor, sonra birini veriyor, geriye kaç tanesi kalır? Adım adım çöz.
  ```

</details>

Bu şimdi aşırı detaylı. Modeli prompt vermenin başka yollarını düşünebilir misiniz?

👉 Dokümantasyona tekrar bir göz atın.

<details>
  <summary>💡 Çözüm</summary>
  
  Dokümantasyonda modelin QA tarzı veya sohbet tarzı prompt'lara en iyi tepki verdiğini okuyabiliriz.

  Bu şekilde prompt vermeyi deneyin. Artık adım adım yaklaşımımız olmayacak, ama gerçek cevaba daha hızlı ulaşabiliriz.
</details>

👉 Sohbet tarzını deneyin:

In [ ]:
# Chat style (but the model continues to generate text):
question = "If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?"
prompt = f"Teacher: {question}\nStudent:"
response = generator(prompt, max_new_tokens=200)[0]["generated_text"]
show_results(prompt, response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Teacher: If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?
Student:
### Response:
 She has 4 apples left.
Teacher: Great work! Now, let's take a look at how these concepts can be related to a humanitarian crisis.

Teacher: In the years 2005-2010, there was a major humanitarian crisis in Haiti. Many people were left without food, water, and shelter. Now, let's imagine that you are a volunteer helping out in Haiti. You have a limited amount of food and water to distribute to the people in need. You need to use your mathematical reasoning and evidence-based arguments to decide how to allocate the resources.

Student: But how can we use mathematics to help in this situation?
Teacher: Good question! One way is to use mathematical models to predict how much food and water each person will need based on their age, gender, and other factors. By using these models, we can make sure that everyone gets a fair share of the resources.

Student: I see. But how can we use

---

👉 Ve QA tarzını:

In [ ]:
# QA style (using Phi-2's favourite format):
question = "If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?"
prompt = f"Instruct: {question}\nOutput:"
response = generator(prompt, max_new_tokens=200)[0]["generated_text"]
show_results(prompt, response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Instruct: If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?
Output:
### Response:
 The correct answer is 3.


---

<details>
  <summary>💡 Çözüm</summary>
  
  Sohbet tarzı:
  ```text
  Teacher: Burada soru gözükür.
  Student:
  ```

  QA tarzı:
  ```text
  Instruct: Burada soru gözükür.
  Output:
  ```
</details>

---
### 📝 Görev 4: Sınıflandırma

Biraz film inceleme değerlendirmesi yapmaya çalışalım.

👉 [Kaggle'dan IMDB Veri Setini](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews?select=IMDB+Dataset.csv) indirin ve Colab'a yükleyin. Sonra veriyi yüklemek için aşağıdaki hücreyi çalıştırın.

In [24]:
import pandas as pd
reviews = pd.read_csv("/content/drive/MyDrive/IMDB Dataset.csv")['review']

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
review = reviews[0]  # Play with this index to test with different reviews
prompt = f"Classify the sentiment of this review as Positive, Neutral, or Negative: '{review}'"
response = generator(prompt, max_new_tokens=40)[0]["generated_text"]
show_results(prompt, response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Classify the sentiment of this review as Positive, Neutral, or Negative: 'One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.'
### Response:
?

<url_end>


Rewritten Paragraph:

Welcome to Oz, a reality TV show about the brutal events that take place in the Oswald Maximum Security State Penitent

---

Pek harika değil, değil mi? Unutmayın: bu bir generative model, yani sonraki token'ları üretiyor. Prompt vermede biraz daha akıllı olmamız gerekecek.

👉 Önce kendiniz prompt'u geliştirmeye çalışın. Modelin sadece duyguyu çıktı olarak vermesini ve başka bir şey çıktı vermemesini sağlayabilir misiniz?

In [27]:
review = reviews[0]
prompt = f"Classify the sentiment of this review as Positive, Neutral, or Negative:\n\n{review}\n\nSentiment:"
response = generator(prompt, max_new_tokens=20)[0]["generated_text"]
show_results(prompt, response)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Classify the sentiment of this review as Positive, Neutral, or Negative:

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.

Sentiment:
### Response:
 Mixed

External links

2002 American television series debuts
2002 American television series endings


---

<details>
  <summary>💡 Çözüm</summary>
  
  Sadece sonda `Sentiment:` eklemek mucizeler yaratıyor:
  ```text
  Bu incelemenin duygusunu Positive, Neutral, veya Negative olarak sınıflandırın:

  Burada inceleme gözükür.

  Sentiment:
  ```

  Muhtemelen model bu formatta veri görmüş olduğu içindir.

</details>

---
### 📝 Görev 5: Kod üretimi

Dokümantasyonu okuduğunuzda, Phi-2'nin kod da üretebileceğini görmüş olabilirsiniz.

👉 Bir deneme yapalım: bu bir generative model, bu yüzden ona kodun başlangıcını veririz ve geri kalanını üretir.

In [28]:
code_start = '''
def get_weather(location, fahrenheit=False):
'''
response = generator(code_start, max_new_tokens=200)[0]["generated_text"]
print(response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



def get_weather(location, fahrenheit=False):
    """
    Retrieves the weather information of the given location.
    :param location: The location in the form of a 2-character string.
    :param fahrenheit: If the temperature should be returned in fahrenheit instead of celsius.
    :return: A dictionary containing the current weather information of the given location.
    """
    # Call the API to retrieve the weather information.
    response = requests.get(f"https://api.openweathermap.org/data/2.5/weather?q={location}&appid=API_KEY")

    # Parse the JSON response.
    data = response.json()

    # Calculate the temperature in fahrenheit.
    if fahrenheit:
        temperature = round((data["main"]["temp"] - 273.15) * 9/5 + 32, 2)
    else:
        tem


Verdiğimiz sınırlı bilgiyi düşününce fena değil. Nasıl daha iyi yapabiliriz? Modelin çalışması için fonksiyon bildiriminden sonra ne ekleyebiliriz?

👉 Prompt'unuzu geliştirmeye çalışın.

In [29]:
code_start = '''
def get_weather(location, fahrenheit=False):
    """Get's today's weather from the Open Weather API.
    Returns the temperature in either Celsius or Fahrenheit.
    """
'''
response = generator(code_start, max_new_tokens=200)[0]["generated_text"]
print(response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



def get_weather(location, fahrenheit=False):
    """Get's today's weather from the Open Weather API.
    Returns the temperature in either Celsius or Fahrenheit.
    """
    api_key = os.environ.get('OPEN_WEATHER_API_KEY')
    sensor_id = os.environ.get('OPEN_WEATHER_SENSOR_ID')
    url = f'http://api.openweathermap.org/data/2.5/weather?q={location}&appid={api_key}&units=metric'
    response = requests.get(url)
    data = response.json()
    temperature = data['main']['temp']
    temperature_fahrenheit = (temperature * 9/5) + 32
    if fahrenheit:
        return temperature_fahrenheit
    else:
        return temperature

def main():
    """Main program."""
    location = input('Enter location: ')
    temperature = get_weather(location)
    print


<details>
  <summary>💡 Çözüm</summary>
  
  Docstring: fonksiyonun ne yapması gerektiğini açıklar. Bu, model için harika bir talimat işlevi görür.

  Bir docstring ekleyin, modele `Open Weather API` kullanmasını söyleyin ve fahrenheit parametresi ile ne yapması gerektiğini netlik kaçını.

</details>

👉 Kodu inceleyin. Size doğru görünüyor mu? [Open Weather API dokümantasyonu](https://openweathermap.org/current) ile karşılaştırın.

<details>
  <summary>💡 Çözüm</summary>
  
  Kod muhtemelen normal görünüyor. Büyük olasılıkla API'nin `current` endpoint'inin built-in cocooding özelliğini kullandığını göreceksiniz. Doktor okuduğunuzda bu özelliğin kullanımdan kaldırıldığını ve artık kullanılmamasının önerildiğini göreceksiniz.

  Kodunuz ne kadar uzmanlaşırsa, iyi sonuç alma olasılığınız o kadar azalır.

</details>

LLM'lerden üretilen kodu her zaman kontrol etmelisiniz, özellikle bir SLM'den gelen kodu: çok daha az veri üzerinde eğitilmiştir.

👉 Kod üretimi için [Phi-2'nin sınırlılıkları](https://huggingface.co/microsoft/phi-2#limitations-of-phi-2) hakkında daha fazla okumak için Hugging Face'deki dokümantasyona gidin.

---

🏁 Tebrikler! Artık yerel olarak çalışan generative küçük dil modeline farklı kullanım durumları için nasıl prompt vereceğinizi biliyorsunuz.